# N-EMD Phase 1 Demo: Synthetic Signals & Classical EMD

This notebook demonstrates:
1. **Synthetic signal generation** — multi-component AM-FM signals with known ground truth
2. **Classical EMD decomposition** — using PyEMD as our baseline
3. **Differentiable building blocks** — Hilbert transform, instantaneous frequency, envelopes
4. **Evaluation metrics** — orthogonality, energy preservation, mode mixing

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({
    "font.family": "serif",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "figure.figsize": (12, 3),
    "figure.dpi": 120,
})

from nemd.utils import (
    generate_synthetic_signal,
    generate_chirp,
    orthogonality_index,
    energy_ratio,
    reconstruction_error,
    mode_mixing_index,
    monotonicity_score,
)
from nemd.classical import ClassicalEMD, VMD
from nemd.layers import (
    hilbert_transform,
    instantaneous_amplitude,
    instantaneous_frequency,
    spectral_bandwidth,
    envelope_mean,
    upper_lower_envelopes,
)

print("All imports OK")

## 1. Synthetic Signal Generation

We create a 3-component AM-FM signal with known ground truth. Each component has a distinct carrier frequency (50, 15, 3 Hz), amplitude modulation, and frequency modulation.

In [ ]:
# Generate a 3-component AM-FM signal
t, signal, components = generate_synthetic_signal(
    n_samples=1024,
    duration=1.0,
    components=[
        {"f0": 50.0, "f_mod": 2.0, "a_mod": 0.5},
        {"f0": 15.0, "f_mod": 0.5, "a_mod": 0.3},
        {"f0": 3.0,  "f_mod": 0.1, "a_mod": 0.2},
    ],
    noise_std=0.05,
    seed=42,
)

# Plot individual components and composite signal
fig, axes = plt.subplots(4, 1, figsize=(12, 8), sharex=True)
labels = ["50 Hz component", "15 Hz component", "3 Hz component"]
for i, (comp, label) in enumerate(zip(components, labels)):
    axes[i].plot(t, comp, linewidth=0.8)
    axes[i].set_ylabel(label)
    axes[i].set_xlim(0, 1)

axes[3].plot(t, signal, linewidth=0.8, color="black")
axes[3].set_ylabel("Composite signal")
axes[3].set_xlabel("Time (s)")
fig.suptitle("Synthetic 3-Component AM-FM Signal", fontweight="bold")
fig.tight_layout()
plt.show()

## 2. Classical EMD Decomposition

We apply standard EMD (Huang et al., 1998) to decompose the composite signal. The classical sifting process extracts IMFs from highest to lowest frequency.

In [ ]:
# Classical EMD
emd = ClassicalEMD()
imfs = emd.decompose(signal, t)

print(f"Number of IMFs (incl. residual): {imfs.shape[0]}")

# Plot IMFs
n_imfs = imfs.shape[0]
fig, axes = plt.subplots(n_imfs + 1, 1, figsize=(12, 2.2 * (n_imfs + 1)), sharex=True)

axes[0].plot(t, signal, "k", linewidth=0.8)
axes[0].set_ylabel("Signal")
axes[0].set_title("Classical EMD Decomposition", fontweight="bold")

for i in range(n_imfs):
    label = f"IMF {i+1}" if i < n_imfs - 1 else "Residual"
    axes[i + 1].plot(t, imfs[i], linewidth=0.8)
    axes[i + 1].set_ylabel(label)

axes[-1].set_xlabel("Time (s)")
fig.tight_layout()
plt.show()

## 3. Evaluation Metrics

Quantitative assessment of the EMD decomposition quality.

In [ ]:
# Compute metrics
oi = orthogonality_index(imfs)
er = energy_ratio(signal, imfs)
re = reconstruction_error(signal, imfs)
mmi = mode_mixing_index(components, imfs)
ms = monotonicity_score(imfs[-1])  # residual

print("=== Classical EMD Metrics ===")
print(f"  Orthogonality Index:    {oi:.4f}   (0 = perfectly orthogonal)")
print(f"  Energy Ratio:           {er:.4f}   (1.0 = perfect preservation)")
print(f"  Reconstruction Error:   {re:.6f} (0 = perfect reconstruction)")
print(f"  Mode Mixing Index:      {mmi:.4f}   (0 = clean separation)")
print(f"  Residual Monotonicity:  {ms:.4f}   (1.0 = perfectly monotonic)")

## 4. VMD Comparison

Variational Mode Decomposition (Dragomiretskiy & Zosso, 2014) requires specifying the number of modes K in advance, but avoids the sequential sifting problems of EMD.

In [ ]:
# VMD with K=3 (matching true number of components)
vmd = VMD(n_modes=3)
modes = vmd.decompose(signal)

fig, axes = plt.subplots(4, 1, figsize=(12, 8), sharex=True)
axes[0].plot(t, signal, "k", linewidth=0.8)
axes[0].set_ylabel("Signal")
axes[0].set_title("VMD Decomposition (K=3)", fontweight="bold")

for i in range(3):
    axes[i + 1].plot(t, modes[i], linewidth=0.8)
    axes[i + 1].set_ylabel(f"Mode {i+1}")
axes[-1].set_xlabel("Time (s)")
fig.tight_layout()
plt.show()

# VMD metrics
oi_vmd = orthogonality_index(modes)
er_vmd = energy_ratio(signal, modes)
re_vmd = reconstruction_error(signal, modes)
mmi_vmd = mode_mixing_index(components, modes)

print("=== VMD Metrics ===")
print(f"  Orthogonality Index:    {oi_vmd:.4f}")
print(f"  Energy Ratio:           {er_vmd:.4f}")
print(f"  Reconstruction Error:   {re_vmd:.6f}")
print(f"  Mode Mixing Index:      {mmi_vmd:.4f}")

## 5. Differentiable Building Blocks

These are the PyTorch-native operations that will power the N-EMD neural network. All support gradient computation.

In [ ]:
# Demonstrate Hilbert transform + instantaneous amplitude on a single component
t_torch = torch.from_numpy(t).float()
comp0 = torch.from_numpy(components[0]).float()

env = instantaneous_amplitude(comp0)
inst_freq = instantaneous_frequency(comp0, fs=1024.0)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 5), sharex=False)

# Envelope
ax1.plot(t, components[0], linewidth=0.6, alpha=0.7, label="50 Hz component")
ax1.plot(t, env.numpy(), "r-", linewidth=1.5, label="Inst. amplitude (envelope)")
ax1.plot(t, -env.numpy(), "r-", linewidth=1.5)
ax1.set_ylabel("Amplitude")
ax1.set_title("Hilbert Transform: Instantaneous Amplitude", fontweight="bold")
ax1.legend(loc="upper right")
ax1.set_xlim(0, 1)

# Instantaneous frequency
t_mid = 0.5 * (t[:-1] + t[1:])
ax2.plot(t_mid, inst_freq.numpy(), linewidth=0.8)
ax2.set_ylabel("Frequency (Hz)")
ax2.set_xlabel("Time (s)")
ax2.set_title("Instantaneous Frequency of 50 Hz AM-FM Component", fontweight="bold")
ax2.axhline(50.0, color="gray", linestyle="--", alpha=0.5, label="Carrier (50 Hz)")
ax2.legend()
ax2.set_xlim(0, 1)
ax2.set_ylim(30, 70)

fig.tight_layout()
plt.show()

In [ ]:
# Differentiable envelope estimation
sig_torch = torch.from_numpy(signal).float()
upper, lower = upper_lower_envelopes(sig_torch, window_size=31)
mean_env = envelope_mean(sig_torch, window_size=51)

fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(t, signal, "k", linewidth=0.6, alpha=0.6, label="Signal")
ax.plot(t, upper.numpy(), "r-", linewidth=1.2, label="Upper envelope")
ax.plot(t, lower.numpy(), "b-", linewidth=1.2, label="Lower envelope")
ax.plot(t, mean_env.numpy(), "g--", linewidth=1.5, label="Mean envelope")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Amplitude")
ax.set_title("Differentiable Envelope Estimation", fontweight="bold")
ax.legend(loc="upper right")
ax.set_xlim(0, 1)
fig.tight_layout()
plt.show()

In [ ]:
# Spectral bandwidth comparison: each component should be narrow-band
bw_values = []
for i, comp in enumerate(components):
    comp_t = torch.from_numpy(comp).float()
    bw = spectral_bandwidth(comp_t, fs=1024.0).item()
    bw_values.append(bw)
    print(f"Component {i+1} (f0={[50, 15, 3][i]} Hz): spectral bandwidth = {bw:.2f} Hz")

# Compare with the composite signal
bw_composite = spectral_bandwidth(sig_torch, fs=1024.0).item()
print(f"Composite signal:                spectral bandwidth = {bw_composite:.2f} Hz")
print("\n(Narrow bandwidth = mono-component = good IMF candidate)")

## 6. Gradient Verification

Critical for N-EMD: all building blocks must be differentiable. We verify that gradients flow through the Hilbert transform and envelope operations.

In [ ]:
# Verify gradient flow through all building blocks
x = torch.randn(256, requires_grad=True)

# Hilbert transform
z = hilbert_transform(x)
loss_hilbert = z.abs().sum()
loss_hilbert.backward()
print(f"Hilbert transform:      grad norm = {x.grad.norm().item():.4f}  (non-zero = OK)")

x.grad.zero_()

# Instantaneous frequency
freq = instantaneous_frequency(x, fs=1.0)
loss_freq = freq.abs().sum()
loss_freq.backward()
print(f"Instantaneous frequency: grad norm = {x.grad.norm().item():.4f}  (non-zero = OK)")

x.grad.zero_()

# Envelope mean
env = envelope_mean(x, window_size=15)
loss_env = env.sum()
loss_env.backward()
print(f"Envelope mean:          grad norm = {x.grad.norm().item():.4f}  (non-zero = OK)")

x.grad.zero_()

# Spectral bandwidth
bw = spectral_bandwidth(x)
bw.backward()
print(f"Spectral bandwidth:     grad norm = {x.grad.norm().item():.4f}  (non-zero = OK)")

print("\nAll building blocks are differentiable!")